This jupyter notebook enables an error propagation based on landmarks.
The inputs encompass the landmarks and transformation matrices.
The landmarks were selected in napari using the affinder plug-in which also determined the transformation matrices (column Matrices in the input table).
In the rows where the ROI is called 'vis' the Matrix refers to the affine transformation matrix between a 10X H&E of the visium and the AFM section.
In all other rows the Matrix refers to the affine transformation matrix between the 20X Hoechst image and the 20X H&E image of the AFM section.
The template matching matrices were determined using the "Mapping_withConfigFile.ipynb" script.

For Patient 1, there was a rotation of 18 degrees between the 20X H&E image and the 10X H&E tilescan of the AFM section. To accomodate this, a rotation matrix was multiplied with
the template matchingmatrices determined by the "Mapping_withConfigFile.ipynb" script.

rotation_matrix = np.array([[ 9.51056516e-01,  3.09016994e-01, -1.26665615e+03],

                            [-3.09016994e-01,  9.51056516e-01,  1.40164784e+03],
       
                            [ 0.00000000e+00,  0.00000000e+00,  1.00000000e+00]])

This correction is already reflected in the input table.

In [73]:
# import libraries and define functions
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import openpyxl
from pathlib import Path
from sklearn.neighbors import NearestNeighbors

# extract of landmarks from excel file
def transform_point_entry_to_array_list(df,roi,index_name):
    landmarks = df[df['ROI']==roi][index_name].tolist()
    landmarks = [np.fromstring(pts.strip('[]\n'),sep=' ') for pts in landmarks]
    return landmarks

# extract matrix from excel file
def transform_matrix_entry_to_matrix(df,roi,column):
    matrix = df[df['ROI']==roi][column].tolist()[0]
    matrix = np.fromstring(matrix.replace('[','').replace(']',''),sep=' ').reshape(3,3)

    #index array to re-order the matrix so that it can be applied to points instead of images
    index_array = np.array([3,2,1,0,5,4])
    return np.hstack([matrix[:2,:2].reshape(-1),matrix[:2,2]])[index_array]

def transform_matrix_entry_to_affine_transform_matrix(df,roi,column):
    matrix = df[df['ROI']==roi][column].tolist()[0]
    matrix = np.fromstring(matrix.replace('[','').replace(']',''),sep=' ').reshape(3,3)
    return matrix

# calculate the covariance ('error') of the matrix parameter
def calculate_parameter_covariance(pts_src,residuals):
    pts_src = np.asarray(pts_src)
    residuals = np.asarray(residuals)
    n = pts_src.shape[0]

    #create Jacobi matrix from partial derivatives
    #x' = ax + by + e
    #y' = cy + dy + f
    # even rows are derivatives of x, odd rows are derivatives of y
    # the columns differ in the variable (a-f) from which they were derived
    J = np.zeros((2*n, 6))
    for i, (xs, ys) in enumerate(pts_src):
        J[2*i]   = [xs, ys, 0, 0, 1, 0]
        J[2*i+1] = [0, 0, xs, ys, 0, 1]
    dof = max(1, 2*n - 6)  # degrees of freedom
    sigma2_hat = (residuals @ residuals) / dof
    # Covariance of parameters: sigma2 * (J^T J)^{-1}
    # Use pseudo-inverse in case J^T J near-singular
    JTJ = J.T.dot(J)
    inv_JTJ = np.linalg.pinv(JTJ)
    Sigma_p = sigma2_hat * inv_JTJ
    return Sigma_p

# transform a point into specific format for calculation
def Jp_affine_at_point(x):
    x1, x2 = x
    Jp = np.array([[x1, x2, 0, 0, 1, 0],
                   [0, 0, x1, x2, 0, 1]])
    return Jp

# Propagate a point trough all transforms and also propagate the error along with it
def propagate_point_A_to_D(xA, affine1, Sigma_p1, translation, affine2, Sigma_p2):
    ## define matrices
    # first affine transformation
    A1 = affine1[:4].reshape(2,2)
    t1 = affine1[4:]
    # translation
    Ts = translation[:4].reshape(2,2)
    ts = translation[4:]
    # second affine transformation
    A2 = affine2[:4].reshape(2,2)
    t2 = affine2[4:]

    ## propagate the point
    # first transform
    xB = A1.dot(xA) + t1
    # second transform
    xC = Ts.dot(xB) + ts
    # third transform
    xD = A2.dot(xC) + t2

    ## propagate the error
    # first transform
    Jp1 = Jp_affine_at_point(xA)
    Sigma_B = Jp1.dot(Sigma_p1).dot(Jp1.T)
    # second transform (no propagation only scaling)
    Sigma_C = Ts.dot(Sigma_B).dot(Ts.T)
    # third transform
    # uncertainty just for last last transform
    Jp2 = Jp_affine_at_point(xC)
    Sigma_from_CD_params = Jp2.dot(Sigma_p2).dot(Jp2.T)
    # actual propagation
    Sigma_from_prev = A2.dot(Sigma_C).dot(A2.T)
    # combined error
    Sigma_D = Sigma_from_CD_params + Sigma_from_prev
    return xD, Sigma_D

def affine6_to_matrix3x3(arr):
    """Convert 6-element affine array to 3x3 matrix."""
    a, b, c, d, tx, ty = arr
    return np.array([
        [a, b, tx],
        [c, d, ty],
        [0, 0, 1]
    ])

In [74]:
# import excel files with landmarks and transformation matrices and points to transform

path_landmarks = Path('../Inputs/Landmarks.xlsx')
data_landmarks = pd.read_excel(path_landmarks)
data_landmarks = data_landmarks.iloc[:-1]
path_afm_points = Path('../Inputs/AFM_Coords.xlsx')
data_afm_points = pd.read_excel(path_afm_points)

# import the AFM mapping table (all patients combined) to merge the propagated error to stiffness values
path_measurements = Path('../Inputs/ST_AFM_Mapping.xlsx')
mapped_df = pd.read_excel(path_measurements)

In [75]:
# get the different samples
samples = data_landmarks['Sample'].unique()

In [76]:
# iterate over each sample and roi to get matrix parameter covariance
cov_p = dict()
matrices = dict()
tm_matrices = dict()
for sample in samples:
    sub_cov_p = dict()
    sub_matrices = dict()
    sub_translation = dict()
    df = data_landmarks[data_landmarks['Sample'] == sample]
    rois = df['ROI'].unique()
    for roi in rois:
        matrix = transform_matrix_entry_to_matrix(df,roi,'Matrix')
        pts_src = transform_point_entry_to_array_list(df,roi,'to_be_transformed')
        residuals = transform_point_entry_to_array_list(df,roi,'difference_single')
        residuals = np.hstack(residuals)
        sub_cov_p[roi] = calculate_parameter_covariance(pts_src,residuals)
        sub_matrices[roi] = matrix
        if roi != 'vis':
            matrix = transform_matrix_entry_to_matrix(df,roi,'tm_matrix')
            sub_translation[roi] = matrix
    cov_p[sample] = sub_cov_p
    matrices[sample] = sub_matrices
    tm_matrices[sample] = sub_translation

In [77]:
# perform transform the coordinates and propagate the error
df = []
for sample in samples:
    afm_df = data_afm_points[data_afm_points['Sample']== sample]
    rois = afm_df['ROI'].unique().tolist()
    for roi in rois:
        coords = transform_point_entry_to_array_list(afm_df,roi,'AFM_ROI_coords')
        #keep the position index for latter mapping to the stiffness values
        position_indices = afm_df[afm_df['ROI'] == roi]['Position_Index'].to_list() 
        for xA, pi in zip(coords, position_indices):
            affine1 = matrices[sample][roi]
            Sigma_p1 = cov_p[sample][roi]
            translation = tm_matrices[sample][roi]
            affine2 = matrices[sample]['vis']
            Sigma_p2 = cov_p[sample]['vis']
            xt,sigma_D = propagate_point_A_to_D(xA, affine1, Sigma_p1, translation, affine2, Sigma_p2)
            std_x, std_y = np.sqrt(np.diag(sigma_D))
            df.append(pd.DataFrame({'Sample':[sample],'ROI':[roi],
                                    'Position_Index': pi,
                                    'x_start':xA[0],'y_start':xA[1],
                                    'x_final':[xt[0]],'y_final':[xt[1]],'std_x':[std_x],'std_y':[std_y]}))
df = pd.concat(df)
df.head()

,Sample,ROI,Position_Index,x_start,y_start,x_final,y_final,std_x,std_y
0,Patient01,ROI01,0,1293.703326,1293.703326,8318.764646,5674.291095,20.308610,20.315291
0,Patient01,ROI01,1,1293.703326,1233.769254,8332.606856,5652.763587,20.427974,20.433518
0,Patient01,ROI01,2,1293.703326,1173.835181,8346.449066,5631.236078,20.549676,20.554229
0,Patient01,ROI01,3,1293.703326,1113.901109,8360.291276,5609.708570,20.673675,20.677383
0,Patient01,ROI01,4,1293.703326,1053.967036,8374.133486,5588.181061,20.799931,20.802935


In [78]:
mapped_df.head()

,Position_Index,X_Position,Y_Position,Baseline_Offset_[N],Baseline_Slope_[N/m],Contact_Point_Offset_[m],Young's_modulus_[Pa],Contact_Point_[m],Baseline_[N],Discontinuity_Position,...,Adhesion_[N],ResidualRMS_[N],ROI,Sample,Spot X,Spot Y,AFM X,AFM Y,Distance_um,Barcode
0,0,-0.00020,-0.00089,9.260000e-09,-0.000140,0.000074,179.0900,5.560000e-08,7.600000e-13,NaN,...,0,9.620000e-12,ROI01,Patient01,8285,5658,8318.868102,5674.563143,27.654422,AGTACGGGCACCTGGC-1
1,1,-0.00018,-0.00089,-2.000000e-09,0.000010,0.000069,72.2305,1.300000e-07,-1.100000e-13,NaN,...,0,9.980000e-12,ROI01,Patient01,8285,5658,8332.711156,5653.034383,35.185845,AGTACGGGCACCTGGC-1
2,2,-0.00016,-0.00089,2.680000e-09,-0.000057,0.000073,14.9237,4.130000e-07,-3.800000e-12,NaN,...,0,8.080000e-12,ROI01,Patient01,8285,5658,8346.554211,5631.505623,49.155707,AGTACGGGCACCTGGC-1
3,3,-0.00014,-0.00089,4.720000e-09,-0.000089,0.000072,41.7625,1.690000e-07,1.880000e-13,NaN,...,0,9.720000e-12,ROI01,Patient01,8406,5591,8360.397265,5609.976863,36.230933,CGCCATCCGATTATGA-1
4,4,-0.00012,-0.00089,1.960000e-08,-0.000300,0.000071,911.0730,-7.200000e-09,2.900000e-11,NaN,...,0,4.080000e-11,ROI01,Patient01,8406,5591,8374.240319,5588.448103,23.371260,CGCCATCCGATTATGA-1


In [79]:
# merge the error propageted and the AFM measurement df
columns_to_keep = ['Sample', 'ROI', 'Position_Index', "Young's_modulus_[Pa]", 'ResidualRMS_[N]', 'Spot X', 'Spot Y', 'AFM X', 'AFM Y',
       'Distance_um', 'Barcode', 'x_final', 'y_final', 'std_x', 'std_y']
merged_df = pd.merge(df, mapped_df, on=['Sample', 'ROI', 'Position_Index'])[columns_to_keep] 

#To continue the original AFM coordinates from the Mapping_withConfigFile script were considered and not the ones from the stepwise transformation in this script.
#For transparency, the difference in mapping is calculated. It is below one pixel and due to rounding differences.

merged_df['x_dif'] = merged_df['AFM X'] - merged_df['x_final']
merged_df['y_dif'] = merged_df['AFM Y'] - merged_df['y_final']

#Print out mapping differences
print('The maximum mapping differences between scripts is ' + str(max(merged_df['x_dif'].max(), merged_df['y_dif'].max())) + ' pixels.')
print('The average mapping difference between scripts in x is: ' + str(merged_df['x_dif'].mean()))
print('The average mapping difference between scripts in y is: ' + str(merged_df['y_dif'].mean()))

The maximum mapping differences between scripts is 0.2790782666816085 pixels.
The average mapping difference between scripts in x is: -0.016859005052020183
The average mapping difference between scripts in y is: -0.008707177083168949


In [80]:
merged_df

,Sample,ROI,Position_Index,Young's_modulus_[Pa],ResidualRMS_[N],Spot X,Spot Y,AFM X,AFM Y,Distance_um,Barcode,x_final,y_final,std_x,std_y,x_dif,y_dif
0,Patient01,ROI01,0,179.0900,9.620000e-12,8285,5658,8318.868102,5674.563143,27.654422,AGTACGGGCACCTGGC-1,8318.764646,5674.291095,20.308610,20.315291,0.103457,0.272048
1,Patient01,ROI01,1,72.2305,9.980000e-12,8285,5658,8332.711156,5653.034383,35.185845,AGTACGGGCACCTGGC-1,8332.606856,5652.763587,20.427974,20.433518,0.104301,0.270796
2,Patient01,ROI01,2,14.9237,8.080000e-12,8285,5658,8346.554211,5631.505623,49.155707,AGTACGGGCACCTGGC-1,8346.449066,5631.236078,20.549676,20.554229,0.105145,0.269545
3,Patient01,ROI01,3,41.7625,9.720000e-12,8406,5591,8360.397265,5609.976863,36.230933,CGCCATCCGATTATGA-1,8360.291276,5609.708570,20.673675,20.677383,0.105989,0.268293
4,Patient01,ROI01,4,911.0730,4.080000e-11,8406,5591,8374.240319,5588.448103,23.371260,CGCCATCCGATTATGA-1,8374.133486,5588.181061,20.799931,20.802935,0.106833,0.267042
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4579,Patient7a,ROI02,94,2682.1500,1.826960e-11,9065,7756,9087.951663,7723.219161,29.353058,CATAGTAGCATAGTAG-1,9087.951666,7723.219157,17.290192,17.286205,-0.000003,0.000004
4580,Patient7a,ROI02,95,3445.7900,1.865250e-11,9065,7756,9062.410987,7718.407159,27.640203,CATAGTAGCATAGTAG-1,9062.410990,7718.407155,17.085280,17.081220,-0.000003,0.000004
4581,Patient7a,ROI02,96,20.8264,7.879850e-12,9065,7756,9036.870312,7713.595156,37.326077,CATAGTAGCATAGTAG-1,9036.870315,7713.595152,16.890121,16.885807,-0.000003,0.000004
4582,Patient7a,ROI02,98,340.3170,1.223920e-11,8945,7686,8985.788961,7703.971150,32.694513,CCAATTGAATGTTAAT-1,8985.788963,7703.971146,16.530424,16.525038,-0.000003,0.000004


In [ ]:
# perform k nearest neighbour analysis 

# Determine the σ levels; 1 refers to the standard deviation. We used half the standard deviation.
# Since sigma_levels is a list multiple levels can be compared at the same time.

sigma_levels = [0.5]

# relative shifts to test (center + corners + axes)
shifts = [
    (0, 0),
    (+1, 0), (-1, 0),
    (0, +1), (0, -1),
    (+1, +1), (+1, -1),
    (-1, +1), (-1, -1),
]

knn_results = []

# iterate over section and extract the spot and AFM coords
for sample, df_sample in merged_df.groupby('Sample'):
    afm_coords_stdv = df_sample[['AFM X', 'AFM Y', 'std_x', 'std_y']].reset_index(drop=True)
    afm_coords = df_sample[['AFM X', 'AFM Y']].to_numpy()
    spot_df = df_sample[['Barcode', 'Spot X', 'Spot Y']].drop_duplicates(subset='Barcode').reset_index(drop=True)
    
    #Separate spots and Barcode
    spot_coords = spot_df[['Spot X', 'Spot Y']].to_numpy()
    spot_barcodes = spot_df['Barcode'].to_numpy()

    knn = NearestNeighbors(n_neighbors=1, metric='euclidean') #n_neighbors = 1 to get the closest spot
    knn.fit(spot_coords)

    distances, indices = knn.kneighbors(afm_coords)

    for afm_idx in range(len(afm_coords)):

        spot_idx = indices[afm_idx, 0]

        result = {
            'Sample': sample,
            'AFM X': afm_coords[afm_idx, 0],
            'AFM Y': afm_coords[afm_idx, 1],
            'Barcode': spot_df.loc[spot_idx, 'Barcode'],
            'Spot X': spot_df.loc[spot_idx, 'Spot X'],
            'Spot Y': spot_df.loc[spot_idx, 'Spot Y'],
            'Distance': distances[afm_idx, 0], #distance in pixels
      
            }

        # ---- stability check per sigma level ----
        for sigma in sigma_levels:

            barcodes_found = set()

            for dx_mul, dy_mul in shifts:
                test_point = np.array([[
                    afm_coords[afm_idx][0] + dx_mul * sigma * afm_coords_stdv.loc[afm_idx, 'std_x'],
                    afm_coords[afm_idx][1] + dy_mul * sigma * afm_coords_stdv.loc[afm_idx, 'std_y']
                ]])

                _, idx = knn.kneighbors(test_point)
                barcodes_found.add(spot_barcodes[idx[0, 0]])
                if len(barcodes_found) > 1: #as soon as two spots are found, the loop breaks to save time
                    break


            result[f'Stable_{sigma}sigma'] = (len(barcodes_found) == 1)

        knn_results.append(result)

knn_results_df = pd.DataFrame(knn_results)


In [83]:
knn_results_df

,Sample,AFM X,AFM Y,Barcode,Spot X,Spot Y,Distance,Stable_0.5sigma
0,Patient01,8318.868102,5674.563143,AGTACGGGCACCTGGC-1,8285,5658,37.701274,True
1,Patient01,8332.711156,5653.034383,AGTACGGGCACCTGGC-1,8285,5658,47.968863,True
2,Patient01,8346.554211,5631.505623,AGTACGGGCACCTGGC-1,8285,5658,67.013975,False
3,Patient01,8360.397265,5609.976863,CGCCATCCGATTATGA-1,8406,5591,49.393631,True
4,Patient01,8374.240319,5588.448103,CGCCATCCGATTATGA-1,8406,5591,31.862039,True
...,...,...,...,...,...,...,...,...
4579,Patient7b,6216.741056,5522.674366,GCTGAGGCGTGAGTAT-1,6277,5536,61.714770,True
4580,Patient7b,6228.856807,5545.993990,GCTGAGGCGTGAGTAT-1,6277,5536,49.169572,True
4581,Patient7b,6240.972559,5569.313614,GCTGAGGCGTGAGTAT-1,6277,5536,49.069067,True
4582,Patient7b,6253.088310,5592.633237,GCTGAGGCGTGAGTAT-1,6277,5536,61.474324,True


In [84]:
# merge all dataframes

error_prop_df = pd.merge(merged_df, knn_results_df, on = ['Sample', 'AFM X', 'AFM Y', 'Barcode', 'Spot X', 'Spot Y'])
error_prop_df

,Sample,ROI,Position_Index,Young's_modulus_[Pa],ResidualRMS_[N],Spot X,Spot Y,AFM X,AFM Y,Distance_um,Barcode,x_final,y_final,std_x,std_y,x_dif,y_dif,Distance,Stable_0.5sigma
0,Patient01,ROI01,0,179.0900,9.620000e-12,8285,5658,8318.868102,5674.563143,27.654422,AGTACGGGCACCTGGC-1,8318.764646,5674.291095,20.308610,20.315291,0.103457,0.272048,37.701274,True
1,Patient01,ROI01,1,72.2305,9.980000e-12,8285,5658,8332.711156,5653.034383,35.185845,AGTACGGGCACCTGGC-1,8332.606856,5652.763587,20.427974,20.433518,0.104301,0.270796,47.968863,True
2,Patient01,ROI01,2,14.9237,8.080000e-12,8285,5658,8346.554211,5631.505623,49.155707,AGTACGGGCACCTGGC-1,8346.449066,5631.236078,20.549676,20.554229,0.105145,0.269545,67.013975,False
3,Patient01,ROI01,3,41.7625,9.720000e-12,8406,5591,8360.397265,5609.976863,36.230933,CGCCATCCGATTATGA-1,8360.291276,5609.708570,20.673675,20.677383,0.105989,0.268293,49.393631,True
4,Patient01,ROI01,4,911.0730,4.080000e-11,8406,5591,8374.240319,5588.448103,23.371260,CGCCATCCGATTATGA-1,8374.133486,5588.181061,20.799931,20.802935,0.106833,0.267042,31.862039,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4579,Patient7a,ROI02,94,2682.1500,1.826960e-11,9065,7756,9087.951663,7723.219161,29.353058,CATAGTAGCATAGTAG-1,9087.951666,7723.219157,17.290192,17.286205,-0.000003,0.000004,40.017024,True
4580,Patient7a,ROI02,95,3445.7900,1.865250e-11,9065,7756,9062.410987,7718.407159,27.640203,CATAGTAGCATAGTAG-1,9062.410990,7718.407155,17.085280,17.081220,-0.000003,0.000004,37.681888,True
4581,Patient7a,ROI02,96,20.8264,7.879850e-12,9065,7756,9036.870312,7713.595156,37.326077,CATAGTAGCATAGTAG-1,9036.870315,7713.595152,16.890121,16.885807,-0.000003,0.000004,50.886641,True
4582,Patient7a,ROI02,98,340.3170,1.223920e-11,8945,7686,8985.788961,7703.971150,32.694513,CCAATTGAATGTTAAT-1,8985.788963,7703.971146,16.530424,16.525038,-0.000003,0.000004,44.572430,True


In [87]:
#only keep stable mapped AFM measurements
filtered_df = error_prop_df[error_prop_df['Stable_0.5sigma'] == True] #column name needs to be changed if the sigma level was changed above
filtered_df


,Sample,ROI,Position_Index,Young's_modulus_[Pa],ResidualRMS_[N],Spot X,Spot Y,AFM X,AFM Y,Distance_um,Barcode,x_final,y_final,std_x,std_y,x_dif,y_dif,Distance,Stable_0.5sigma
0,Patient01,ROI01,0,179.0900,9.620000e-12,8285,5658,8318.868102,5674.563143,27.654422,AGTACGGGCACCTGGC-1,8318.764646,5674.291095,20.308610,20.315291,0.103457,0.272048,37.701274,True
1,Patient01,ROI01,1,72.2305,9.980000e-12,8285,5658,8332.711156,5653.034383,35.185845,AGTACGGGCACCTGGC-1,8332.606856,5652.763587,20.427974,20.433518,0.104301,0.270796,47.968863,True
3,Patient01,ROI01,3,41.7625,9.720000e-12,8406,5591,8360.397265,5609.976863,36.230933,CGCCATCCGATTATGA-1,8360.291276,5609.708570,20.673675,20.677383,0.105989,0.268293,49.393631,True
4,Patient01,ROI01,4,911.0730,4.080000e-11,8406,5591,8374.240319,5588.448103,23.371260,CGCCATCCGATTATGA-1,8374.133486,5588.181061,20.799931,20.802935,0.106833,0.267042,31.862039,True
5,Patient01,ROI01,5,118.0150,1.270000e-11,8406,5591,8388.083373,5566.919343,22.016227,CGCCATCCGATTATGA-1,8387.975696,5566.653553,20.928402,20.930842,0.107677,0.265790,30.014722,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4579,Patient7a,ROI02,94,2682.1500,1.826960e-11,9065,7756,9087.951663,7723.219161,29.353058,CATAGTAGCATAGTAG-1,9087.951666,7723.219157,17.290192,17.286205,-0.000003,0.000004,40.017024,True
4580,Patient7a,ROI02,95,3445.7900,1.865250e-11,9065,7756,9062.410987,7718.407159,27.640203,CATAGTAGCATAGTAG-1,9062.410990,7718.407155,17.085280,17.081220,-0.000003,0.000004,37.681888,True
4581,Patient7a,ROI02,96,20.8264,7.879850e-12,9065,7756,9036.870312,7713.595156,37.326077,CATAGTAGCATAGTAG-1,9036.870315,7713.595152,16.890121,16.885807,-0.000003,0.000004,50.886641,True
4582,Patient7a,ROI02,98,340.3170,1.223920e-11,8945,7686,8985.788961,7703.971150,32.694513,CCAATTGAATGTTAAT-1,8985.788963,7703.971146,16.530424,16.525038,-0.000003,0.000004,44.572430,True


In [89]:
#Define output path and save the dfs

# Save the mapped and filtered dataframes. Existing files will be overwritten.
output_path = Path('../Outputs')

error_prop_df.to_csv(
   output_path / 'Error_Propagated_Data.csv', index=False)

filtered_df.to_csv(
   output_path / 'Mapped_Filtered_AFM_Measurements.csv', index=False)
